# 네이버페이 증권 종목토론실 데이터 크롤링 - 2
---
이전에 [네이버페이 증권 종목토론실 데이터 크롤링](https://github.com/boringariel/python/blob/tmp/crawling/%EB%84%A4%EC%9D%B4%EB%B2%84%ED%8E%98%EC%9D%B4%20%EC%A6%9D%EA%B6%8C%20%EC%A2%85%EB%AA%A9%ED%86%A0%EB%A1%A0%EC%8B%A4%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%ED%81%AC%EB%A1%A4%EB%A7%81.ipynb) 을 통해서 종목토론실 데이터를 판다스 데이터프레임(Pandas DataFrame) 형태로 받아오는 방법을 간단하게 알아보았습니다. 이 데이터를 그대로 사용해도 좋지만, 본문 데이터를 함께 사용하고 싶은 분들이라면 각 게시글의 URL 정보를 함께 크롤링하는게 좋습니다. 그래서, 이번에는 파이썬(Python)과 뷰티풀수프(BeautifulSoup)를 사용하는 것은 같지만, 게시글 URL 및 본문 데이터를 함께 크롤링할 수 있도록 코드를 변경해 보겠습니다.  
</p></br></br>

## 데이터 크롤링
---
이번에는 URL 정보를 받아오기 위해, html 문서의 a 태그 매개변수를 탐색해야 합니다. 그래서, `pandas.read_html()` 함수를 사용한 파싱에 더해서, URL이 필요한 행에 한해 `BeautifulSoup.find_all()` 함수를 활용해 table 태그를 찾은 뒤, 반복문으로 각 정보에 접근해야 합니다.  
</p>
이 방법 역시, 판다스를 이용한 방법과 같이 두 번째 표가 종목 토론실 정보이므로 리턴된 리스트의 1번을 인덱싱해 줍니다.  
</p></br></br>


In [1]:
# Import pacakge
import requests
from bs4 import BeautifulSoup
import pandas as pd


CODE = '005930'
PAGE = 1
URL = f"https://finance.naver.com/item/board.naver?code={CODE}&page={PAGE}"

# 접속 정보를 입력
HEADERS = {
    'Referer': 'https://finance.naver.com/item/board.naver?code=005930',
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

response = requests.get(URL, headers=HEADERS)
soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find_all('table')[1]

  
</p></br></br>

### html로 표를 표시하는 방법
---
표를 표시하는 html 태그인 table 태그는 tr, td로 구성되어 있습니다. 이 구조를 알게 된다면, 손쉽게 크롤링에 이용할 반복문을 작성할 수 있습니다.  
</p></br></br>

* tr: table row의 약자로, 한 열을 의미합니다.
* td: table data의 약자로, 하나의 셀을 의미합니다.  
</p></br></br>

<table>
    <tr><td>-></td><td>이 방향으로 한 줄이 tr 태그로 묶여 있습니다 </td></tr>
    <tr><td>1</td><td>이 셀 하나가 td 태그로 묶여 있습니다</td></tr>
    <tr><td>2</td><td>이 셀도 td 태그입니다</td></tr>
</table>  
</p></br></br>

만약 위 표가 table이라는 변수로 저장되어 있고, 이 표의 첫 번째 행(->, 1, 2)을 출력하려고 한다면 아래 코드를 적어서 해결할 수 있습니다.  
</p></br></br>

```python
for _row in table.find_all('tr'):
    print(_row.find('td').text)
```  
</p></br></br>

### a 태그의 매개변수에 접근하는 방법
---
특정 URL에 접속하기 위해서 흔히 사용되는 html 태그로는 a 태그가 있습니다. a 태그는 href라는 매개변수에 접속할 URL을 기록하는데요, 예를 들어서 https://www.google.com 으로 접속하도록 하고 싶다면, 아래와 같은 태그를 작성하면 됩니다.  
</p></br></br>

`<a href="https://www.google.com">이 글을 클릭하면 구글로 이동합니다</a>`  
</p></br></br>

이제, 위 방법을 활용해서 네이버페이 증권 종목토론실의 게시글 URL 데이터를 크롤링해 보겠습니다. 각 열의 정보에서 a 태그가 있다면, 해당 태그의 href 매개변수를 인덱싱해 저장하도록 합니다. 이 때, 기록된 URL은 'https://finance.naver.com' 주소가 생략된 값이므로, 해당 데이터를 추가해 주도록 합니다.  
</p></br></br>


In [2]:
links = []
for row in table.find_all("tr")[1:]:  # 첫 번째 행(헤더) 제외
    a_tag = row.find("a")

    if a_tag:
        link = 'https://finance.naver.com' + str(a_tag['href'])
    else:
        link = None

    links.append(link)

</p></br></br>

이제, 위 데이터를 이용해서 기존 표 데이터에 URL 정보를 더해주도록 하면 됩니다. 기존 판다스 데이터프레임에 URL이라는 이름으로 행을 추가하도록 합니다. URL이 없는 데이터는 결측치(None)으로 입력되도록 코드를 작성했기 때문에, 모든 정보가 결측치인 열만 지워지도록 `dropna()` 함수에 `how='all'` 이라는 매개변수를 추가해 주도록 합니다. 
</p></br></br>


In [3]:
stock_table = pd.read_html(response.text)[1]
stock_table['URL'] = links
stock_table = stock_table.iloc[3:].drop('Unnamed: 6',axis=1).dropna(how='all')
stock_table.head()

,날짜,제목,글쓴이,조회,공감,비공감,URL
3,2025.02.25 09:36,클린봇이 이용자 보호를 위해 숨긴 게시글입니다.,shul****,2,0,0,None
4,2025.02.25 09:36,ㅡ한국 수준은 중국과 얼추비슷 하다 ㅡ ...,true****,2,0,0,https://finance.naver.com/item/board_read.nave...
5,2025.02.25 09:35,낼 엔비디아 실적발표후 쪽박찬다,hohy****,4,0,0,https://finance.naver.com/item/board_read.nave...
6,2025.02.25 09:35,클린봇이 이용자 보호를 위해 숨긴 게시글입니다.,ji******,4,2,0,None
7,2025.02.25 09:35,윤석열 사형집행 후 토리특식,iuit****,7,3,0,https://finance.naver.com/item/board_read.nave...
